# JobHunter Codebase Notebook

This notebook contains the full codebase snapshot, organized by file.


## `app.py`


In [ ]:
"""
JobHunter - Persistent Candidate Search Assistant
Main Streamlit application entry point.

Run with: streamlit run app.py
"""

from __future__ import annotations

import os
import sys
import uuid
from collections import Counter

import streamlit as st
from dotenv import load_dotenv

# Add project root to path
sys.path.insert(0, os.path.dirname(__file__))

load_dotenv()

from src.gmail_client import fetch_rejection_emails
from src.graph import build_graph
from src.chat_agent import ask_agent
from src.profile_builder import create_or_update_user_profile, load_all_profiles
from ui.applications_panel import render_applications_panel
from ui.components import inject_custom_css
from ui.jobs_panel import render_jobs_panel
from ui.strategy_panel import render_strategy_panel


st.set_page_config(
    page_title="JobHunter - Career Copilot",
    page_icon="JH",
    layout="wide",
    initial_sidebar_state="expanded",
)

inject_custom_css()


if "graph" not in st.session_state:
    graph, conn = build_graph()
    st.session_state.graph = graph
    st.session_state.db_conn = conn

if "new_applications" not in st.session_state:
    st.session_state.new_applications = []

if "manual_feedback" not in st.session_state:
    st.session_state.manual_feedback = []

if "strategy_approved" not in st.session_state:
    st.session_state.strategy_approved = False

if "strategy_rejected" not in st.session_state:
    st.session_state.strategy_rejected = False

if "agent_ran" not in st.session_state:
    st.session_state.agent_ran = False

if "persistent_state" not in st.session_state:
    st.session_state.persistent_state = {}
if "agent_chat_history" not in st.session_state:
    st.session_state.agent_chat_history = []


def load_profiles() -> dict:
    return load_all_profiles()


def run_agent(selected_key: str, profiles_data: dict):
    """Run the graph with merged state and persist the result."""
    graph = st.session_state.graph
    profile = profiles_data["profiles"][selected_key]
    default_strategy = profiles_data.get("default_strategies", {}).get(selected_key, {})

    prev = dict(st.session_state.persistent_state)
    prev_apps = prev.get("applications", [])
    new_apps = list(st.session_state.new_applications)

    pending_feedback = ""
    if st.session_state.manual_feedback:
        pending_feedback = "\n---\n".join(fb["text"] for fb in st.session_state.manual_feedback if fb.get("text"))

    input_state = {
        "candidate_profile": profile,
        "applications": prev_apps + new_apps,
        "feedback_log": prev.get("feedback_log", []),
        "rejection_patterns": prev.get("rejection_patterns", []),
        "current_strategy": prev.get("current_strategy") or default_strategy,
        "session_count": prev.get("session_count", 0),
        "sessions_count": prev.get("sessions_count", prev.get("session_count", 0)),
        "matched_jobs": [],
        "new_applications": new_apps,
        "pending_feedback_text": pending_feedback,
        "strategy_approved": st.session_state.strategy_approved,
        "strategy_proposal": prev.get("strategy_proposal", {}),
        "kpi_metrics": prev.get("kpi_metrics", {}),
        "strategy_change_log": prev.get("strategy_change_log", []),
        "usefulness_ratings": prev.get("usefulness_ratings", []),
    }

    thread_id = f"{selected_key}_{uuid.uuid4().hex[:8]}"
    config = {"configurable": {"thread_id": thread_id}}

    with st.spinner("Running agent analysis..."):
        result = graph.invoke(input_state, config=config)

    st.session_state.persistent_state = dict(result)
    st.session_state.agent_ran = True
    st.session_state.new_applications = []
    st.session_state.manual_feedback = []
    st.session_state.strategy_approved = False
    st.session_state.strategy_rejected = False


def render_overview(state: dict):
    """Render high-level progress and next-step guidance."""
    applications = state.get("applications", [])
    kpi = state.get("kpi_metrics", {})
    patterns = state.get("rejection_patterns", [])
    proposal = state.get("strategy_proposal", {})

    if not kpi:
        counts = Counter(app.get("status", "applied") for app in applications)
        total = len(applications)
        interview_eligible = counts.get("interviewing", 0) + counts.get("offer", 0)
        offer_rate = round((counts.get("offer", 0) / interview_eligible) * 100, 1) if interview_eligible else 0.0
        kpi = {
            "total_applications": total,
            "interviewing": counts.get("interviewing", 0),
            "interview_conversion_rate": round((interview_eligible / total) * 100, 1) if total else 0,
            "offer": counts.get("offer", 0),
            "offer_rate": offer_rate,
        }

    queued_apps = len(st.session_state.get("new_applications", []))
    queued_feedback = len(st.session_state.get("manual_feedback", []))

    m1, m2, m3, m4, m5, m6, m7 = st.columns(7)
    m1.metric("Applications", kpi.get("total_applications", 0))
    m2.metric("Interviewing", kpi.get("interviewing", 0))
    m3.metric("Interview Rate", f"{kpi.get('interview_conversion_rate', 0)}%")
    m4.metric("Offers", kpi.get("offer", 0))
    m5.metric("Pending Inputs", queued_apps + queued_feedback)
    m6.metric("Offer Rate", f"{kpi.get('offer_rate', 0)}%")
    avg_days = kpi.get("avg_days_to_outcome")
    m7.metric("Avg Days to Outcome", "N/A" if avg_days is None else avg_days)

    next_steps: list[str] = []
    if not state.get("matched_jobs"):
        next_steps.append("Run agent to generate the first batch of ranked jobs.")
    if queued_apps:
        next_steps.append(f"Run agent to ingest {queued_apps} newly added application(s).")
    if queued_feedback:
        next_steps.append(f"Run agent to parse {queued_feedback} pending feedback item(s).")
    if proposal and proposal.get("theme"):
        next_steps.append("Review the pending strategy pivot in Strategy Lab.")
    if patterns:
        next_steps.append("Use detected patterns to update prep focus this week.")

    left, right = st.columns([1.3, 1])
    with left:
        st.markdown('<div class="section-header">Next Best Actions</div>', unsafe_allow_html=True)
        if next_steps:
            for step in next_steps:
                st.markdown(f'<div class="next-step">{step}</div>', unsafe_allow_html=True)
        else:
            st.success("No urgent actions pending. The pipeline is up to date.")

        recent_apps = applications[-5:]
        if recent_apps:
            st.markdown("**Recent Applications**")
            st.dataframe(
                [
                    {
                        "Company": a.get("company_name", ""),
                        "Role": a.get("role_title", ""),
                        "Status": a.get("status", ""),
                        "Applied": a.get("date_applied", ""),
                    }
                    for a in reversed(recent_apps)
                ],
                use_container_width=True,
                hide_index=True,
            )

    with right:
        strategy = state.get("current_strategy", {})
        st.markdown('<div class="section-header">Current Direction</div>', unsafe_allow_html=True)
        st.markdown(
            f"""
            <div class="panel-card">
                <div style="font-size:12px; color:#64748B;">Target seniority</div>
                <div style="font-size:22px; font-weight:700; color:#0F172A; margin-top:4px;">
                    {strategy.get('target_seniority', 'Senior')}
                </div>
                <div style="font-size:12px; color:#64748B; margin-top:10px;">
                    Sessions completed: {state.get('session_count', state.get('sessions_count', 0))}
                </div>
                <div style="font-size:12px; color:#64748B; margin-top:4px;">
                    Detected patterns: {len(patterns)}
                </div>
            </div>
            """,
            unsafe_allow_html=True,
        )


def render_agent_chat(profile: dict, state: dict):
    st.markdown('<div class="section-header">Agent Chat</div>', unsafe_allow_html=True)
    st.markdown(
        '<div class="sub-header">Talk with your JobHunter copilot about strategy, applications, and next moves.</div>',
        unsafe_allow_html=True,
    )

    if not os.environ.get("GOOGLE_API_KEY", "").strip():
        st.warning("Backend LLM key missing: set GOOGLE_API_KEY in .env and restart app.")

    for msg in st.session_state.agent_chat_history:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    user_message = st.chat_input("Ask your copilot anything about your job search...")
    if user_message:
        st.session_state.agent_chat_history.append({"role": "user", "content": user_message})
        with st.chat_message("user"):
            st.markdown(user_message)

        with st.chat_message("assistant"):
            with st.spinner("Thinking..."):
                reply = ask_agent(user_message, profile, state)
            st.markdown(reply)
        st.session_state.agent_chat_history.append({"role": "assistant", "content": reply})


profiles_data = load_profiles()
profile_options = {key: prof["name"] for key, prof in profiles_data["profiles"].items()}

with st.sidebar:
    st.markdown(
        """
        <div class="brand-wrap">
            <div class="brand-mark"></div>
            <div class="brand-title">JobHunter</div>
            <div class="brand-sub">AI Career Copilot</div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    if not profile_options:
        st.error("No profiles available. Create one below to continue.")

    # Apply deferred profile switch BEFORE the selectbox is created.
    # Streamlit forbids mutating a widget's session_state key after widget instantiation.
    pending_profile_switch = st.session_state.get("next_profile_key")
    if pending_profile_switch in profile_options:
        st.session_state.profile_selector = pending_profile_switch
        st.session_state.last_profile = pending_profile_switch
        st.session_state.persistent_state = {}
        st.session_state.new_applications = []
        st.session_state.manual_feedback = []
        del st.session_state["next_profile_key"]

    selected_key = st.selectbox(
        "Candidate profile",
        options=list(profile_options.keys()),
        format_func=lambda x: f"{profile_options[x]} ({profiles_data['profiles'][x].get('target_role', 'Role')})",
        key="profile_selector",
    )

    if "last_profile" not in st.session_state:
        st.session_state.last_profile = selected_key
    elif st.session_state.last_profile != selected_key:
        st.session_state.persistent_state = {}
        st.session_state.last_profile = selected_key
        st.session_state.new_applications = []
        st.session_state.manual_feedback = []

    selected_profile = profiles_data["profiles"].get(selected_key, {})
    skill_count = len(selected_profile.get("base_skills", []))
    st.caption(f"Profile skills tracked: {skill_count}")
    if selected_profile.get("target_salary"):
        st.caption(f"Target salary: {selected_profile.get('target_salary')}")

    with st.expander("Create Personalized Profile", expanded=False):
        st.caption("Upload CV and related documents to build a richer candidate profile.")

        with st.form("create_profile_form", clear_on_submit=False):
            create_name = st.text_input("Full name")
            create_role = st.text_input("Target role", value="Software Engineer")
            create_years = st.number_input("Years of experience", min_value=0, max_value=50, value=3, step=1)
            create_salary = st.text_input("Target salary (optional)", placeholder="$140k-$180k")
            create_notes = st.text_area(
                "Job search context",
                placeholder="Preferred industries, locations, salary range, job types, achievements...",
                height=90,
            )
            uploaded_files = st.file_uploader(
                "Upload CV / resume / job documents",
                type=["txt", "md", "csv", "json", "yaml", "yml", "log", "pdf"],
                accept_multiple_files=True,
            )

            submitted = st.form_submit_button("Create profile", use_container_width=True)

        if submitted:
            if not create_name.strip():
                st.warning("Please add a profile name.")
            else:
                with st.spinner("Building profile from your documents..."):
                    new_key, _, _, warnings = create_or_update_user_profile(
                        name=create_name,
                        target_role=create_role,
                        years_experience=int(create_years),
                        target_salary=create_salary,
                        additional_notes=create_notes,
                        uploaded_files=uploaded_files or [],
                        gemini_api_key=os.environ.get("GOOGLE_API_KEY", ""),
                    )

                for warning in warnings:
                    st.info(warning)

                st.success("Profile created. Switched to your new profile.")
                st.session_state.next_profile_key = new_key
                st.rerun()

    state_snapshot = dict(st.session_state.persistent_state)
    session_count = state_snapshot.get("session_count", state_snapshot.get("sessions_count", 0))
    st.caption(f"Sessions completed: {session_count}")

    if st.button("Run Agent", use_container_width=True, type="primary"):
        try:
            run_agent(selected_key, profiles_data)
            st.success("Agent run complete.")
            st.rerun()
        except Exception as exc:
            st.error(f"Agent error: {exc}")

    st.markdown("---")
    st.markdown("### Workflow")
    steps = [
        f"Jobs ranked: {len(state_snapshot.get('matched_jobs', []))}",
        f"Tracked applications: {len(state_snapshot.get('applications', []))}",
        f"Pending new applications: {len(st.session_state.new_applications)}",
        f"Pending feedback notes: {len(st.session_state.manual_feedback)}",
    ]
    for step in steps:
        st.markdown(f"- {step}")

    st.markdown("---")
    st.markdown("### Feedback Intake")

    gmail_available = os.path.exists(os.path.join(os.path.dirname(__file__), "credentials.json"))
    if gmail_available:
        if st.button("Scan Gmail", use_container_width=True):
            with st.spinner("Scanning inbox..."):
                emails = fetch_rejection_emails(max_results=5)
            if emails:
                for email_text in emails:
                    st.session_state.manual_feedback.append({"app_id": "", "text": email_text})
                st.success(f"Added {len(emails)} email(s) to pending feedback.")
            else:
                st.info("No relevant emails found.")
            st.rerun()
    else:
        st.info("No Gmail credentials found. Use manual feedback entry.")

    manual_text = st.text_area(
        "Manual feedback",
        placeholder="Paste rejection text or interview notes.",
        key="sidebar_feedback",
        height=110,
    )
    if st.button("Queue Feedback", use_container_width=True):
        if manual_text.strip():
            st.session_state.manual_feedback.append({"app_id": "", "text": manual_text.strip()})
            st.success("Feedback queued.")
            st.rerun()
        else:
            st.warning("Add some text before queueing feedback.")

    with st.expander("Demo quick actions"):
        demo_rejections = [
            "We decided to move forward with candidates who demonstrated stronger system design skills.",
            "The system design round did not meet expectations for this role.",
            "Your system design approach did not align with our architecture standards.",
            "Coding skills were strong, but system design depth was below our bar.",
            "We are prioritizing candidates with broader systems experience.",
        ]

        if st.button("Add 3 system design rejections", use_container_width=True):
            for text in demo_rejections[:3]:
                st.session_state.manual_feedback.append({"app_id": "", "text": text})
            st.success("Added 3 demo feedback entries.")
            st.rerun()

        if st.button("Add 2 additional rejections", use_container_width=True):
            for text in demo_rejections[3:5]:
                st.session_state.manual_feedback.append({"app_id": "", "text": text})
            st.success("Added 2 demo feedback entries.")
            st.rerun()


current_state = dict(st.session_state.persistent_state)
selected_profile = profiles_data["profiles"][selected_key]

st.markdown(
    f"""
    <div style="padding: 4px 0 16px 0;">
        <h1 style="margin: 0; font-size: 30px; color: #0F172A;">JobHunter Workspace</h1>
        <p style="margin: 4px 0 0 0; color: #475569; font-size: 14px;">
            Candidate: <strong>{selected_profile.get('name', '')}</strong> | Target role: {selected_profile.get('target_role', '')}
        </p>
        <p style="margin: 2px 0 0 0; color: #64748B; font-size: 12px;">
            Target salary: {selected_profile.get('target_salary', 'Not set')}
        </p>
    </div>
    """,
    unsafe_allow_html=True,
)


overview_tab, jobs_tab, apps_tab, strategy_tab, chat_tab = st.tabs(
    [
        "Overview",
        "Recommended Jobs",
        "Applications",
        "Agent Strategy Log",
        "Agent Chat",
    ]
)

with overview_tab:
    render_overview(current_state)

with jobs_tab:
    render_jobs_panel(current_state)

with apps_tab:
    render_applications_panel(current_state)

with strategy_tab:
    render_strategy_panel(current_state)

with chat_tab:
    render_agent_chat(selected_profile, current_state)


st.markdown("---")
st.caption("JobHunter v2 flow: Overview -> Jobs -> Applications -> Strategy")


## `data/demo_profiles.json`


In [ ]:
{
  "profiles": {
    "alex_demo": {
      "name": "Alex Chen",
      "base_skills": ["Python", "React", "AWS", "Docker", "PostgreSQL", "REST APIs", "Git", "CI/CD", "System Design", "Microservices"],
      "target_role": "Senior Software Engineer",
      "years_experience": 5
    },
    "jordan_demo": {
      "name": "Jordan Rivera",
      "base_skills": ["JavaScript", "HTML", "CSS", "React", "Node.js", "Git", "REST APIs"],
      "target_role": "Frontend Developer",
      "years_experience": 2
    },
    "sam_demo": {
      "name": "Sam Patel",
      "base_skills": ["Java", "Spring Boot", "Kubernetes", "AWS", "MySQL", "Kafka", "Terraform", "Python"],
      "target_role": "Backend Engineer",
      "years_experience": 4
    }
  },
  "default_strategies": {
    "alex_demo": {
      "target_seniority": "Senior",
      "focus_areas": ["Python", "React", "AWS"],
      "prep_recommendations": [],
      "locked_changes": [],
      "positive_events_since_last_change": 0
    },
    "jordan_demo": {
      "target_seniority": "Mid-Level",
      "focus_areas": ["JavaScript", "React"],
      "prep_recommendations": [],
      "locked_changes": [],
      "positive_events_since_last_change": 0
    },
    "sam_demo": {
      "target_seniority": "Senior",
      "focus_areas": ["Java", "Spring Boot", "AWS"],
      "prep_recommendations": [],
      "locked_changes": [],
      "positive_events_since_last_change": 0
    }
  }
}


## `data/user_profiles.json`


In [ ]:
{
  "profiles": {
    "sashank_user": {
      "name": "sashank",
      "base_skills": [],
      "target_role": "AI engineer",
      "years_experience": 1,
      "notes": ""
    },
    "sbguhdvs_user": {
      "name": "sbguhdvs",
      "base_skills": [],
      "target_role": "Software Engineer",
      "years_experience": 1,
      "target_salary": "140",
      "notes": "AI automations, workflows"
    }
  },
  "default_strategies": {
    "sashank_user": {
      "target_seniority": "Junior",
      "focus_areas": [],
      "prep_recommendations": [],
      "locked_changes": [],
      "positive_events_since_last_change": 0
    },
    "sbguhdvs_user": {
      "target_seniority": "Junior",
      "focus_areas": [],
      "prep_recommendations": [],
      "locked_changes": [],
      "positive_events_since_last_change": 0
    }
  }
}

## `README.md`


In [ ]:
# JobHunter: Persistent Candidate Search Assistant

## Overview
JobHunter is a stateful, long-running career copilot built with LangGraph and Streamlit. It tracks the full application lifecycle across sessions, learns from feedback, detects rejection patterns, and proposes strategy pivots.

## Problem
High-volume job seekers apply to many roles weekly, but most workflows are spreadsheet-driven and static. Rejection feedback is rarely synthesized into actionable strategy changes, and profile setup across tools is repetitive.

## Solution
JobHunter acts as a persistent memory agent:
1. Magic Onboarding: CV/document upload with LLM extraction into a candidate profile, strategy baseline, and target salary.
2. Dynamic Matching: Job recommendations from a local CSV dataset using evolving strategy and fit scoring.
3. Automated Tracking: Gmail integration for application-related emails and rejection parsing.
4. Strategy Pivot: Pattern detection over historical rejection data with human approval before strategy updates.
5. Agent Chat: Built-in LLM chat so users can communicate directly with their copilot.

## Target Users
- High-volume tech candidates applying to 10+ roles per week.
- Career pivoters calibrating role seniority.
- Candidates repeatedly blocked in specific interview stages.

## MVP Scope
- CV/doc onboarding with PDF + text ingestion.
- Multi-session persistence through LangGraph SqliteSaver.
- Stable local job feed from CSV/Kaggle-style dataset.
- Gmail OAuth integration for inbox scanning.
- Pattern detection threshold of 3+ historical feedback items.
- Core dashboard panels:
  - Job Feed
  - Application Tracker
  - Agent Strategy Log

## Tech Stack

### Core Logic & Agent Framework
- LangGraph (state machine + persistent checkpointer)
- Pydantic v2 (typed state models)
- LangChain + Gemini (LLM orchestration)

### Data Ingestion
- PyPDF2 (CV PDF text extraction)

### Persistence & Storage
- SQLite via LangGraph SqliteSaver (state memory)
- Local CSV + Pandas (job database)
- JSON profile stores (demo + user profiles)

### Frontend
- Streamlit

### Integrations
- Gmail API (OAuth 2.0)

## Data Storage
- Demo profiles: `data/demo_profiles.json`
- User-created profiles: `data/user_profiles.json`
- Graph checkpoint/state: `checkpoints/jobhunter.db`
- Env configuration: `.env` (including `GOOGLE_API_KEY`)

## Running Locally
1. Install dependencies:
   - `pip install -r requirements.txt`
2. Add backend key to `.env`:
   - `GOOGLE_API_KEY=...`
3. Start app:
   - `streamlit run app.py`

## Notes
- Manual API-key entry is intentionally removed from UI.
- LLM features (profile extraction + chat) use backend environment configuration only.


## `requirements.txt`


In [ ]:
langgraph>=0.2.0
langgraph-checkpoint-sqlite>=1.0.0
langchain>=0.3.0
langchain-google-genai>=2.0.0
streamlit>=1.38.0
pandas>=2.0.0
pydantic>=2.0.0
python-dotenv>=1.0.0
google-api-python-client>=2.100.0
google-auth-httplib2>=0.2.0
google-auth-oauthlib>=1.2.0
PyPDF2>=3.0.0


## `src/__init__.py`


In [ ]:
# JobHunter source package


## `src/chat_agent.py`


In [ ]:
"""
LLM chat helper for direct user-agent communication inside the app.
Uses backend environment key only (GOOGLE_API_KEY).
"""

from __future__ import annotations

import os
from typing import Any


def _build_context(profile: dict[str, Any], state: dict[str, Any]) -> str:
    strategy = state.get("current_strategy", {})
    kpi = state.get("kpi_metrics", {})
    patterns = state.get("rejection_patterns", [])
    apps = state.get("applications", [])

    return (
        f"Candidate name: {profile.get('name', 'Unknown')}\n"
        f"Target role: {profile.get('target_role', 'Unknown')}\n"
        f"Years experience: {profile.get('years_experience', 0)}\n"
        f"Base skills: {', '.join(profile.get('base_skills', []))}\n"
        f"Current target seniority: {strategy.get('target_seniority', 'Unknown')}\n"
        f"Focus areas: {', '.join(strategy.get('focus_areas', []))}\n"
        f"Prep recommendations: {', '.join(strategy.get('prep_recommendations', []))}\n"
        f"Tracked applications: {len(apps)}\n"
        f"Interview conversion rate: {kpi.get('interview_conversion_rate', 0)}%\n"
        f"Offer rate: {kpi.get('offer_rate', 0)}%\n"
        f"Detected patterns: {', '.join(p.get('theme', '') for p in patterns) if patterns else 'None'}"
    )


def ask_agent(message: str, profile: dict[str, Any], state: dict[str, Any]) -> str:
    """Generate assistant reply for in-app chat."""
    api_key = os.environ.get("GOOGLE_API_KEY", "").strip()
    if not api_key:
        return (
            "Backend LLM key is not configured. Add GOOGLE_API_KEY to .env on the server, "
            "restart the app, and chat will work automatically."
        )

    try:
        from langchain_google_genai import ChatGoogleGenerativeAI

        llm = ChatGoogleGenerativeAI(
            model="gemini-2.0-flash",
            google_api_key=api_key,
            temperature=0.2,
        )

        system_prompt = (
            "You are JobHunter Copilot, a practical career assistant. "
            "Give specific, concise, actionable advice based on the candidate context. "
            "If asked for plans, provide clear next steps."
        )
        context = _build_context(profile, state)
        prompt = (
            f"{system_prompt}\n\n"
            f"Candidate context:\n{context}\n\n"
            f"User question:\n{message}\n\n"
            "Answer:"
        )
        response = llm.invoke(prompt)
        text = response.content if hasattr(response, "content") else str(response)
        return (text or "").strip() or "I could not generate a response. Please try again."
    except Exception as exc:
        return f"Agent chat error: {exc}"


## `src/gmail_client.py`


In [ ]:
"""
Gmail API client for fetching rejection/update emails.
Uses OAuth 2.0 with Desktop app credentials.
Gracefully falls back to empty results if credentials are unavailable.
"""

from __future__ import annotations

import os
import base64
from typing import List, Optional


def get_gmail_service():
    """
    Authenticate with Gmail API using OAuth 2.0.
    Requires credentials.json in the project root.
    Returns the Gmail API service object, or None if unavailable.
    """
    try:
        from googleapiclient.discovery import build
        from google_auth_oauthlib.flow import InstalledAppFlow
        from google.auth.transport.requests import Request
        from google.oauth2.credentials import Credentials

        SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]
        creds = None

        project_root = os.path.join(os.path.dirname(__file__), "..")
        token_path = os.path.join(project_root, "token.json")
        creds_path = os.path.join(project_root, "credentials.json")

        if not os.path.exists(creds_path):
            return None

        if os.path.exists(token_path):
            creds = Credentials.from_authorized_user_file(token_path, SCOPES)

        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            else:
                flow = InstalledAppFlow.from_client_secrets_file(creds_path, SCOPES)
                creds = flow.run_local_server(port=0)

            with open(token_path, "w") as token:
                token.write(creds.to_json())

        return build("gmail", "v1", credentials=creds)

    except Exception as e:
        print(f"Gmail API setup failed: {e}")
        return None


def fetch_rejection_emails(
    max_results: int = 10,
    sender_filter: Optional[str] = None,
) -> List[str]:
    """
    Fetch email body texts matching job application update patterns.

    Args:
        max_results: Maximum number of emails to fetch.
        sender_filter: Specific sender email to filter by.

    Returns:
        List of email body text strings. Empty list if Gmail is unavailable.
    """
    service = get_gmail_service()
    if service is None:
        return []

    try:
        # Build query
        queries = []
        if sender_filter:
            queries.append(f"from:{sender_filter}")
        else:
            queries.append(
                "(from:no-reply@greenhouse.io OR from:no-reply@lever.co "
                "OR from:notifications@hire.lever.co OR from:noreply@hired.com "
                'OR subject:"application update" OR subject:"application status" '
                'OR subject:"your application" OR subject:"regarding your application")'
            )

        query = " ".join(queries)

        results = service.users().messages().list(
            userId="me",
            q=query,
            maxResults=max_results,
        ).execute()

        messages = results.get("messages", [])
        email_texts = []

        for msg in messages:
            full_msg = service.users().messages().get(
                userId="me",
                id=msg["id"],
                format="full",
            ).execute()

            body_text = _extract_body(full_msg)
            if body_text:
                email_texts.append(body_text)

        return email_texts

    except Exception as e:
        print(f"Gmail fetch failed: {e}")
        return []


def _extract_body(message: dict) -> str:
    """Extract plain text body from a Gmail API message."""
    try:
        payload = message.get("payload", {})

        # Simple message
        if payload.get("body", {}).get("data"):
            return base64.urlsafe_b64decode(payload["body"]["data"]).decode("utf-8", errors="replace")

        # Multipart message
        parts = payload.get("parts", [])
        for part in parts:
            if part.get("mimeType") == "text/plain":
                data = part.get("body", {}).get("data", "")
                if data:
                    return base64.urlsafe_b64decode(data).decode("utf-8", errors="replace")

        # Fallback to snippet
        return message.get("snippet", "")

    except Exception:
        return message.get("snippet", "")


## `src/graph.py`


In [ ]:
"""
JobHunter LangGraph State Machine.

Graph flow:
  state_loader → job_fetcher → matcher → ranker → application_tracker
    → (if feedback exists) feedback_parser → pattern_detector
    → (if pattern found, INTERRUPT) strategy_updater → job_fetcher (re-rank)
    → state_saver → END
"""

from __future__ import annotations

import os
import sqlite3

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, StateGraph

from src.models import JobHunterState
from src.nodes.state_loader import state_loader
from src.nodes.job_fetcher import job_fetcher
from src.nodes.matcher import matcher
from src.nodes.ranker import ranker
from src.nodes.application_tracker import application_tracker
from src.nodes.feedback_parser import feedback_parser
from src.nodes.pattern_detector import pattern_detector
from src.nodes.strategy_updater import strategy_updater
from src.nodes.state_saver import state_saver


# ---------------------------------------------------------------------------
# Routing functions
# ---------------------------------------------------------------------------

def should_parse_feedback(state: dict) -> str:
    """Route to feedback_parser if there's pending feedback text.
    Also route to strategy_updater directly if approval is pending."""
    pending = state.get("pending_feedback_text", "")
    approved = state.get("strategy_approved", False)
    proposal = state.get("strategy_proposal", {})

    if pending and pending.strip():
        return "feedback_parser"
    # If approval is pending from a previous session, go directly to strategy_updater
    if approved and proposal and proposal.get("theme"):
        return "strategy_updater"
    return "state_saver"


def should_update_strategy(state: dict) -> str:
    """Route to strategy_updater if a pattern was found and approved."""
    proposal = state.get("strategy_proposal", {})
    approved = state.get("strategy_approved", False)

    if proposal and proposal.get("theme"):
        if approved:
            return "strategy_updater"
        # Pattern found but not yet approved — go to state_saver
        # The UI will handle the interrupt / approval flow
        return "state_saver"
    return "state_saver"


def after_strategy_update(state: dict) -> str:
    """After strategy update, re-fetch jobs with new criteria."""
    return "job_fetcher_refetch"


# ---------------------------------------------------------------------------
# Graph builder
# ---------------------------------------------------------------------------

def build_graph(db_path: str | None = None) -> tuple:
    """
    Build and compile the JobHunter state graph with SqliteSaver persistence.

    Returns:
        (compiled_graph, checkpointer_connection)
    """
    # Set up SQLite checkpointer
    if db_path is None:
        checkpoint_dir = os.path.join(os.path.dirname(__file__), "..", "checkpoints")
        os.makedirs(checkpoint_dir, exist_ok=True)
        db_path = os.path.join(checkpoint_dir, "jobhunter.db")

    conn = sqlite3.connect(db_path, check_same_thread=False)
    memory = SqliteSaver(conn)

    # Build the graph
    builder = StateGraph(JobHunterState)

    # Add all 9 required nodes
    builder.add_node("state_loader", state_loader)
    builder.add_node("job_fetcher", job_fetcher)
    builder.add_node("matcher", matcher)
    builder.add_node("ranker", ranker)
    builder.add_node("application_tracker", application_tracker)
    builder.add_node("feedback_parser", feedback_parser)
    builder.add_node("pattern_detector", pattern_detector)
    builder.add_node("strategy_updater", strategy_updater)
    builder.add_node("state_saver", state_saver)

    # Add a re-fetch node (same function, different name for the cycle)
    builder.add_node("job_fetcher_refetch", job_fetcher)
    builder.add_node("matcher_refetch", matcher)
    builder.add_node("ranker_refetch", ranker)

    # Define edges
    builder.set_entry_point("state_loader")

    # Main flow: state_loader → job_fetcher → matcher → ranker → application_tracker
    builder.add_edge("state_loader", "job_fetcher")
    builder.add_edge("job_fetcher", "matcher")
    builder.add_edge("matcher", "ranker")
    builder.add_edge("ranker", "application_tracker")

    # After tracking apps, check for feedback or pending approval
    builder.add_conditional_edges(
        "application_tracker",
        should_parse_feedback,
        {
            "feedback_parser": "feedback_parser",
            "strategy_updater": "strategy_updater",
            "state_saver": "state_saver",
        },
    )

    # After parsing feedback, detect patterns
    builder.add_edge("feedback_parser", "pattern_detector")

    # After pattern detection, check if strategy should update
    builder.add_conditional_edges(
        "pattern_detector",
        should_update_strategy,
        {
            "strategy_updater": "strategy_updater",
            "state_saver": "state_saver",
        },
    )

    # After strategy update, re-fetch jobs with new criteria
    builder.add_edge("strategy_updater", "job_fetcher_refetch")
    builder.add_edge("job_fetcher_refetch", "matcher_refetch")
    builder.add_edge("matcher_refetch", "ranker_refetch")
    builder.add_edge("ranker_refetch", "state_saver")

    # End
    builder.add_edge("state_saver", END)

    # Compile with checkpointer
    graph = builder.compile(checkpointer=memory)

    return graph, conn


## `src/models.py`


In [ ]:
"""
Pydantic v2 data models for JobHunter state machine.
All models are strictly typed to prevent the AI from hallucinating invalid states.
"""

from __future__ import annotations

import uuid
from datetime import datetime
from enum import Enum
from typing import List, Optional

from pydantic import BaseModel, Field
from typing_extensions import TypedDict


# ---------------------------------------------------------------------------
# Enums
# ---------------------------------------------------------------------------

class ApplicationStatus(str, Enum):
    """The 5 mandatory application statuses."""
    APPLIED = "applied"
    SCREENING = "screening"
    INTERVIEWING = "interviewing"
    REJECTED = "rejected"
    OFFER = "offer"


class Seniority(str, Enum):
    JUNIOR = "Junior"
    MID_LEVEL = "Mid-Level"
    SENIOR = "Senior"


# ---------------------------------------------------------------------------
# Core domain models
# ---------------------------------------------------------------------------

class CandidateProfile(BaseModel):
    """Immutable-ish profile loaded once per candidate."""
    name: str
    base_skills: List[str] = Field(default_factory=list)
    target_role: str = "Software Engineer"
    years_experience: int = 0
    target_salary: str = ""


class SearchStrategy(BaseModel):
    """Dynamic strategy that the agent modifies over sessions."""
    target_seniority: str = Field(
        default="Senior",
        description="e.g., Junior, Mid-Level, Senior",
    )
    focus_areas: List[str] = Field(
        default_factory=list,
        description="Skills to heavily index on",
    )
    prep_recommendations: List[str] = Field(
        default_factory=list,
        description="Resources the user should study based on feedback",
    )
    locked_changes: List[str] = Field(
        default_factory=list,
        description="Strategy changes on cooldown (cannot be reverted for 1 session)",
    )
    positive_events_since_last_change: int = Field(
        default=0,
        description="Count of positive events (screening/interviewing) since last pivot",
    )


class Application(BaseModel):
    """A single job application the candidate has submitted."""
    job_id: str = Field(default_factory=lambda: str(uuid.uuid4())[:8])
    company_name: str = ""
    role_title: str = ""
    status: str = Field(
        default=ApplicationStatus.APPLIED.value,
        description="Must be: applied, screening, interviewing, rejected, offer",
    )
    date_applied: str = Field(
        default_factory=lambda: datetime.now().strftime("%Y-%m-%d"),
    )
    date_last_updated: str = Field(
        default_factory=lambda: datetime.now().strftime("%Y-%m-%d"),
    )
    date_outcome: str = Field(
        default="",
        description="Set when status reaches terminal outcome (rejected/offer).",
    )


class FeedbackEvent(BaseModel):
    """Parsed feedback from a rejection email or manual entry."""
    application_id: str = ""
    raw_email_text: str = ""
    extracted_reason: str = ""
    date_logged: str = Field(
        default_factory=lambda: datetime.now().strftime("%Y-%m-%d"),
    )


class Pattern(BaseModel):
    """A recurring theme detected across multiple rejections."""
    theme: str = Field(description="The recurring issue, e.g., 'System Design gap'")
    occurrences: int = Field(description="Number of times this theme appeared")
    proposed_action: str = Field(
        description="What the agent wants to do about it",
    )


class JobPosting(BaseModel):
    """A job listing from the CSV dataset."""
    job_id: str = ""
    company: str = ""
    title: str = ""
    seniority: str = "Senior"
    description: str = ""
    required_skills: List[str] = Field(default_factory=list)
    fit_score: float = 0.0


# ---------------------------------------------------------------------------
# LangGraph global state
# ---------------------------------------------------------------------------

class JobHunterStateModel(BaseModel):
    """
    Pydantic representation of graph state for validation and requirement alignment.
    LangGraph runtime still uses the TypedDict state below for mutation semantics.
    """

    candidate_profile: CandidateProfile = Field(default_factory=lambda: CandidateProfile(name=""))
    applications: List[Application] = Field(default_factory=list)
    feedback_log: List[FeedbackEvent] = Field(default_factory=list)
    rejection_patterns: List[Pattern] = Field(default_factory=list)
    current_strategy: SearchStrategy = Field(default_factory=SearchStrategy)
    sessions_count: int = 0


class JobHunterState(TypedDict, total=False):
    """The complete graph state persisted by SqliteSaver."""
    candidate_profile: dict  # Serialized CandidateProfile
    applications: list       # List of serialized Application dicts
    feedback_log: list       # List of serialized FeedbackEvent dicts
    rejection_patterns: list # List of serialized Pattern dicts
    current_strategy: dict   # Serialized SearchStrategy
    session_count: int
    sessions_count: int      # Alias kept for requirement compatibility
    matched_jobs: list       # List of serialized JobPosting dicts
    pending_feedback_text: str  # Raw text from Gmail/manual paste awaiting parsing
    strategy_approved: bool     # Whether the user approved a proposed strategy change
    strategy_proposal: dict     # The proposed Pattern + action awaiting approval
    new_applications: list      # New applications added this session
    kpi_metrics: dict           # Computed KPIs for the dashboard
    strategy_change_log: list   # Explainable history of approved strategy updates
    usefulness_ratings: list    # Candidate feedback on pattern-detection usefulness


## `src/nodes/__init__.py`


In [ ]:
# JobHunter nodes package


## `src/nodes/application_tracker.py`


In [ ]:
"""
application_tracker — Records new applications logged by the candidate.
Validates status against the 5 allowed enum values.
"""

from __future__ import annotations

from datetime import datetime

from src.models import ApplicationStatus


VALID_STATUSES = {s.value for s in ApplicationStatus}


def application_tracker(state: dict) -> dict:
    """Process and validate new applications added during this session."""
    existing = state.get("applications", [])
    new_apps = state.get("new_applications", [])

    if not new_apps:
        return {}

    validated = []
    for app in new_apps:
        # Enforce valid status
        status = app.get("status", "applied")
        if status not in VALID_STATUSES:
            status = "applied"

        date_applied = app.get("date_applied", datetime.now().strftime("%Y-%m-%d"))
        validated.append({
            "job_id": app.get("job_id", ""),
            "company_name": app.get("company_name", ""),
            "role_title": app.get("role_title", ""),
            "status": status,
            "date_applied": date_applied,
            "date_last_updated": app.get("date_last_updated", date_applied),
            "date_outcome": app.get("date_outcome", ""),
        })

    # Track positive events for strategy cooldown
    strategy = state.get("current_strategy", {})
    positive_count = strategy.get("positive_events_since_last_change", 0)
    for app in validated:
        if app["status"] in ("screening", "interviewing", "offer"):
            positive_count += 1

    updated_strategy = dict(strategy)
    updated_strategy["positive_events_since_last_change"] = positive_count

    return {
        "applications": existing + validated,
        "new_applications": [],  # Clear after processing
        "current_strategy": updated_strategy,
    }


## `src/nodes/feedback_parser.py`


In [ ]:
﻿"""
feedback_parser - Processes rejection/offer/interview feedback.
Uses LLM to extract a 1-sentence reason from raw text.
Falls back to keyword extraction if LLM is unavailable.
"""

from __future__ import annotations

import os
import re
from datetime import datetime


def _extract_reason_keyword_fallback(raw_text: str) -> str:
    """Simple keyword-based extraction when LLM is unavailable."""
    text_lower = raw_text.lower()

    patterns = {
        "system design": "Candidate lacked system design skills",
        "design round": "Failed the system design interview round",
        "coding challenge": "Did not pass the coding challenge",
        "culture fit": "Not a culture fit for the team",
        "experience": "Insufficient experience for the role",
        "seniority": "Seniority level mismatch",
        "communication": "Communication skills need improvement",
        "technical": "Technical skills did not meet requirements",
        "react": "Lacked required React/frontend experience",
        "python": "Insufficient Python experience",
        "leadership": "Did not demonstrate leadership ability",
        "competitive": "More competitive candidates were selected",
        "salary": "Salary expectations did not align",
        "relocated": "Position was relocated or filled internally",
        "offer accepted": "Candidate received and accepted an offer",
        "congratulations": "Candidate received an offer",
        "next round": "Candidate advanced to next interview round",
        "schedule.*interview": "Candidate scheduled for interview",
    }

    for pattern, reason in patterns.items():
        if re.search(pattern, text_lower):
            return reason

    return "Position filled - moved forward with other candidates"


def _extract_reason_llm(raw_text: str) -> str:
    """Use LLM to extract a 1-sentence rejection reason."""
    try:
        from langchain_google_genai import ChatGoogleGenerativeAI

        api_key = os.environ.get("GOOGLE_API_KEY", "")
        if not api_key:
            return _extract_reason_keyword_fallback(raw_text)

        llm = ChatGoogleGenerativeAI(
            model="gemini-2.0-flash",
            google_api_key=api_key,
            temperature=0,
        )

        prompt = (
            "You are parsing a job application feedback email. "
            "Extract the PRIMARY reason for rejection (or outcome) in exactly ONE sentence. "
            "If it's an offer or interview invitation, state that clearly. "
            "If the reason is unclear, say 'Position filled - moved forward with other candidates'. "
            f"\n\nEmail text:\n{raw_text[:1000]}\n\nOne-sentence reason:"
        )

        response = llm.invoke(prompt)
        reason = response.content.strip()
        return reason if reason else _extract_reason_keyword_fallback(raw_text)

    except Exception:
        return _extract_reason_keyword_fallback(raw_text)


def feedback_parser(state: dict) -> dict:
    """Parse pending feedback text and create FeedbackEvent entries."""
    pending_text = state.get("pending_feedback_text", "")
    if not pending_text or not pending_text.strip():
        return {}

    feedback_log = list(state.get("feedback_log", []))
    applications = list(state.get("applications", []))

    # Split by --- separator to handle multiple feedback entries.
    raw_entries = [t.strip() for t in pending_text.split("---") if t.strip()]

    for raw_text in raw_entries:
        extracted_reason = _extract_reason_llm(raw_text)
        now_str = datetime.now().strftime("%Y-%m-%d")

        reason_lower = extracted_reason.lower()
        if any(w in reason_lower for w in ["offer", "congratulations", "accepted"]):
            new_status = "offer"
        elif any(w in reason_lower for w in ["interview", "next round", "schedule"]):
            new_status = "interviewing"
        elif any(w in reason_lower for w in ["screening", "phone screen", "recruiter"]):
            new_status = "screening"
        else:
            new_status = "rejected"

        # Attach feedback to the most recent in-flight application.
        target_app_id = ""
        for i in range(len(applications) - 1, -1, -1):
            app = applications[i]
            if app.get("status") in ("applied", "screening", "interviewing"):
                target_app_id = app.get("job_id", "")
                applications[i] = dict(app)
                applications[i]["status"] = new_status
                applications[i]["date_last_updated"] = now_str
                if new_status in ("rejected", "offer"):
                    applications[i]["date_outcome"] = now_str
                break

        feedback_event = {
            "application_id": target_app_id,
            "raw_email_text": raw_text[:500],
            "extracted_reason": extracted_reason,
            "date_logged": now_str,
        }
        feedback_log.append(feedback_event)

    return {
        "feedback_log": feedback_log,
        "applications": applications,
        "pending_feedback_text": "",
    }


## `src/nodes/job_fetcher.py`


In [ ]:
"""
job_fetcher — Reads jobs.csv and filters by current strategy's target seniority.
"""

from __future__ import annotations

import os

import pandas as pd

from src.utils import normalize_skills


DATA_DIR = os.path.join(os.path.dirname(__file__), "..", "..", "data")


def job_fetcher(state: dict) -> dict:
    """Fetch jobs from CSV filtered by the current strategy's target seniority."""
    strategy = state.get("current_strategy", {})
    target_seniority = strategy.get("target_seniority", "Senior")
    focus_areas = [s.lower() for s in strategy.get("focus_areas", [])]

    csv_path = os.path.join(DATA_DIR, "jobs.csv")
    df = pd.read_csv(csv_path)

    # Primary filter: match seniority
    filtered = df[df["seniority"].str.strip().str.lower() == target_seniority.strip().lower()]

    # If no matches (shouldn't happen with our dataset), fall back to all
    if filtered.empty:
        filtered = df

    jobs = []
    for _, row in filtered.iterrows():
        skills = normalize_skills(row.get("required_skills", ""))
        jobs.append({
            "job_id": row["job_id"],
            "company": row["company"],
            "title": row["title"],
            "seniority": row["seniority"],
            "description": row.get("description", ""),
            "required_skills": skills,
            "fit_score": 0.0,  # Will be computed by matcher
        })

    return {"matched_jobs": jobs}


## `src/nodes/matcher.py`


In [ ]:
"""
matcher — Scores each job against the candidate profile using keyword overlap.
Applies the seniority penalty per business rules.
"""

from __future__ import annotations

from src.utils import compute_skill_overlap, apply_seniority_penalty


def matcher(state: dict) -> dict:
    """Compute fit scores for all matched jobs."""
    jobs = state.get("matched_jobs", [])
    profile = state.get("candidate_profile", {})
    strategy = state.get("current_strategy", {})

    candidate_skills = profile.get("base_skills", [])
    target_seniority = strategy.get("target_seniority", "Senior")
    focus_areas = [s.lower() for s in strategy.get("focus_areas", [])]

    scored_jobs = []
    for job in jobs:
        job_skills = job.get("required_skills", [])

        # Base score: keyword overlap
        base_score = compute_skill_overlap(job_skills, candidate_skills)

        # Boost for focus area alignment
        focus_overlap = sum(
            1 for skill in job_skills
            if skill.lower() in focus_areas
        )
        focus_bonus = min(focus_overlap * 5, 20)  # Cap at +20

        score = min(base_score + focus_bonus, 100.0)

        # Seniority penalty
        score = apply_seniority_penalty(score, job.get("seniority", ""), target_seniority)

        job_copy = dict(job)
        job_copy["fit_score"] = round(score, 1)
        scored_jobs.append(job_copy)

    return {"matched_jobs": scored_jobs}


## `src/nodes/pattern_detector.py`


In [ ]:
"""
pattern_detector — Detects recurring themes in rejection feedback.
Only triggers if len(feedback_log) >= 3 (business rule).
Uses LLM for clustering with keyword fallback.
"""

from __future__ import annotations

import os
import re
from collections import Counter
from typing import List, Tuple


# Keywords that map to common rejection themes
THEME_KEYWORDS = {
    "System Design": ["system design", "design round", "architecture", "scalability", "distributed systems"],
    "Data Structures & Algorithms": ["algorithm", "data structure", "coding challenge", "leetcode", "dsa"],
    "Frontend Skills": ["react", "frontend", "css", "ui", "hooks", "component"],
    "Backend Skills": ["backend", "api", "database", "server", "microservice"],
    "Communication": ["communication", "culture fit", "team fit", "soft skills", "presentation"],
    "Experience Level": ["experience", "seniority", "years", "overqualified", "underqualified"],
    "Leadership": ["leadership", "management", "lead", "mentoring", "team lead"],
    "Cloud/DevOps": ["aws", "cloud", "devops", "kubernetes", "docker", "infrastructure"],
    "Domain Knowledge": ["domain", "industry", "fintech", "healthcare", "specific knowledge"],
}


def _detect_patterns_keyword(feedback_log: list) -> List[dict]:
    """Keyword-based pattern detection across feedback entries."""
    theme_counts: Counter = Counter()
    theme_reasons: dict = {}

    for event in feedback_log:
        reason = event.get("extracted_reason", "").lower()

        for theme, keywords in THEME_KEYWORDS.items():
            if any(kw in reason for kw in keywords):
                theme_counts[theme] += 1
                if theme not in theme_reasons:
                    theme_reasons[theme] = []
                theme_reasons[theme].append(event.get("extracted_reason", ""))

    patterns = []
    for theme, count in theme_counts.items():
        if count >= 3:  # Business rule: minimum 3 occurrences
            if theme in ("System Design", "Data Structures & Algorithms"):
                action = f"Down-level target seniority to Mid-Level and add {theme} preparation resources"
            elif theme == "Experience Level":
                action = "Adjust target seniority down one level to match market expectations"
            elif theme in ("Frontend Skills", "Backend Skills"):
                action = f"Add {theme.replace(' Skills', '')} courses to prep recommendations and adjust focus areas"
            else:
                action = f"Add {theme} improvement resources to preparation plan"

            patterns.append({
                "theme": theme,
                "occurrences": count,
                "proposed_action": action,
            })

    return patterns


def _detect_patterns_llm(feedback_log: list) -> List[dict]:
    """Use LLM to cluster rejection reasons into themes."""
    try:
        from langchain_google_genai import ChatGoogleGenerativeAI

        api_key = os.environ.get("GOOGLE_API_KEY", "")
        if not api_key:
            return _detect_patterns_keyword(feedback_log)

        llm = ChatGoogleGenerativeAI(
            model="gemini-2.0-flash",
            google_api_key=api_key,
            temperature=0,
        )

        reasons = "\n".join(
            f"- {e.get('extracted_reason', 'Unknown')}" for e in feedback_log
        )

        prompt = (
            "Analyze these job rejection reasons and identify recurring themes. "
            "For each theme that appears 3 or more times, output EXACTLY in this format:\n"
            "THEME: <theme name>\n"
            "COUNT: <number>\n"
            "ACTION: <proposed strategy change>\n\n"
            f"Rejection reasons:\n{reasons}\n\n"
            "Output themes (only those with 3+ occurrences):"
        )

        response = llm.invoke(prompt)
        text = response.content.strip()

        # Parse LLM output
        patterns = []
        theme_blocks = text.split("THEME:")
        for block in theme_blocks[1:]:  # Skip first empty split
            lines = block.strip().split("\n")
            theme = lines[0].strip()
            count = 3
            action = ""
            for line in lines[1:]:
                if line.strip().startswith("COUNT:"):
                    try:
                        count = int(re.search(r"\d+", line).group())
                    except (AttributeError, ValueError):
                        count = 3
                elif line.strip().startswith("ACTION:"):
                    action = line.replace("ACTION:", "").strip()

            if count >= 3 and theme:
                patterns.append({
                    "theme": theme,
                    "occurrences": count,
                    "proposed_action": action or f"Adjust strategy based on {theme} weakness",
                })

        return patterns if patterns else _detect_patterns_keyword(feedback_log)

    except Exception:
        return _detect_patterns_keyword(feedback_log)


def pattern_detector(state: dict) -> dict:
    """Detect recurring rejection patterns. Only runs if >= 3 feedback events."""
    feedback_log = state.get("feedback_log", [])

    # Business rule: need at least 3 events
    if len(feedback_log) < 3:
        return {"rejection_patterns": [], "strategy_proposal": {}}

    # Check cooling rules — skip patterns whose changes are locked
    strategy = state.get("current_strategy", {})
    locked = set(strategy.get("locked_changes", []))
    positive_since_change = strategy.get("positive_events_since_last_change", 0)

    def is_negative_outcome(reason: str) -> bool:
        text = reason.lower()
        positive_signals = ["offer", "congratulations", "advanced to next interview round", "scheduled for interview"]
        negative_signals = [
            "reject",
            "failed",
            "did not",
            "insufficient",
            "lacked",
            "mismatch",
            "not meet",
            "moved forward with other candidates",
            "position filled",
            "weak",
        ]
        if any(n in text for n in negative_signals):
            return True
        if any(p in text for p in positive_signals):
            return False
        # Default unresolved feedback to negative so we can still learn from it.
        return True

    rejections = [e for e in feedback_log if is_negative_outcome(e.get("extracted_reason", ""))]

    if len(rejections) < 3:
        return {"rejection_patterns": [], "strategy_proposal": {}}

    # Try LLM, fall back to keywords
    patterns = _detect_patterns_llm(rejections)

    # Filter out locked patterns (unless cooldown expired via positive events)
    active_patterns = []
    for p in patterns:
        theme_key = p["theme"].lower()
        if theme_key in {lc.lower() for lc in locked} and positive_since_change < 3:
            continue  # Still on cooldown
        active_patterns.append(p)

    if active_patterns:
        # Propose the most significant pattern (highest occurrences)
        top_pattern = max(active_patterns, key=lambda p: p["occurrences"])
        return {
            "rejection_patterns": active_patterns,
            "strategy_proposal": top_pattern,
        }

    return {"rejection_patterns": active_patterns, "strategy_proposal": {}}


## `src/nodes/ranker.py`


In [ ]:
"""
ranker — Sorts scored jobs by fit_score descending, returns top 10.
"""

from __future__ import annotations


def ranker(state: dict) -> dict:
    """Rank jobs by fit score and return the top 10."""
    jobs = state.get("matched_jobs", [])

    # Sort descending by fit_score
    ranked = sorted(jobs, key=lambda j: j.get("fit_score", 0), reverse=True)

    # Return top 10
    top_jobs = ranked[:10]

    return {"matched_jobs": top_jobs}


## `src/nodes/state_loader.py`


In [ ]:
﻿"""
state_loader - Loads candidate profile, application history, and strategy context.
On cold start, it honors explicitly provided profile and strategy from the UI.
"""

from __future__ import annotations


def _infer_default_strategy_from_profile(profile: dict) -> dict:
    years = int(profile.get("years_experience", 0) or 0)
    if years <= 2:
        seniority = "Junior"
    elif years <= 5:
        seniority = "Mid-Level"
    else:
        seniority = "Senior"

    return {
        "target_seniority": seniority,
        "focus_areas": list(profile.get("base_skills", [])[:3]),
        "prep_recommendations": [],
        "locked_changes": [],
        "positive_events_since_last_change": 0,
    }


def state_loader(state: dict) -> dict:
    """Load or initialize candidate state."""
    session_count = state.get("session_count", state.get("sessions_count", 0))
    candidate_profile = state.get("candidate_profile") or {}
    current_strategy = state.get("current_strategy") or {}

    if session_count == 0:
        strategy = current_strategy or _infer_default_strategy_from_profile(candidate_profile)
        return {
            "candidate_profile": candidate_profile,
            "current_strategy": strategy,
            "applications": state.get("applications", []),
            "feedback_log": state.get("feedback_log", []),
            "rejection_patterns": state.get("rejection_patterns", []),
            "session_count": 1,
            "sessions_count": 1,
            "matched_jobs": [],
            "kpi_metrics": state.get("kpi_metrics", {}),
            "strategy_change_log": state.get("strategy_change_log", []),
            "usefulness_ratings": state.get("usefulness_ratings", []),
        }

    return {
        "session_count": session_count + 1,
        "sessions_count": session_count + 1,
        "matched_jobs": [],
    }


## `src/nodes/state_saver.py`


In [ ]:
"""
state_saver — Final node that computes KPI metrics before checkpoint.
LangGraph's SqliteSaver handles the actual persistence automatically.
"""

from __future__ import annotations

from src.utils import compute_kpi_metrics


def state_saver(state: dict) -> dict:
    """Compute final KPIs and prepare state for checkpoint."""
    applications = state.get("applications", [])
    usefulness_ratings = state.get("usefulness_ratings", [])
    metrics = compute_kpi_metrics(applications, usefulness_ratings)

    return {
        "kpi_metrics": metrics,
    }


## `src/nodes/strategy_updater.py`


In [ ]:
﻿"""
strategy_updater - Adjusts the search strategy based on approved patterns.
Only runs after human approval.
"""

from __future__ import annotations

from datetime import datetime


SENIORITY_LEVELS = ["Junior", "Mid-Level", "Senior"]


def _downlevel_seniority(current: str) -> str:
    """Move seniority down one level."""
    current_lower = current.strip().lower()
    for i, level in enumerate(SENIORITY_LEVELS):
        if level.lower() == current_lower and i > 0:
            return SENIORITY_LEVELS[i - 1]
    return current


PREP_RESOURCES = {
    "System Design": [
        "Read 'Designing Data-Intensive Applications' by Martin Kleppmann",
        "Watch System Design Interview channel content",
        "Practice on systemdesign.one or designgurus.io",
    ],
    "Data Structures & Algorithms": [
        "Complete the NeetCode 150 problem set",
        "Read 'Cracking the Coding Interview' by Gayle McDowell",
        "Watch Abdul Bari algorithm lectures",
    ],
    "Frontend Skills": [
        "Review the React Hooks documentation",
        "Build three projects with React and TypeScript",
        "Take a Frontend Masters React course",
    ],
    "Backend Skills": [
        "Read 'Clean Architecture' by Robert Martin",
        "Build one REST API project from scratch",
        "Study backend fundamentals content",
    ],
    "Communication": [
        "Read 'Crucial Conversations'",
        "Practice STAR method responses",
        "Run two mock interview sessions",
    ],
    "Leadership": [
        "Read 'The Manager's Path' by Camille Fournier",
        "Document three leadership examples",
        "Practice staff-level interview questions",
    ],
}


def strategy_updater(state: dict) -> dict:
    """Apply the approved strategy change and persist an explainable change log."""
    strategy = dict(state.get("current_strategy", {}))
    proposal = state.get("strategy_proposal", {})
    approved = state.get("strategy_approved", False)

    if not approved or not proposal:
        return {}

    theme = proposal.get("theme", "Unknown")
    action = proposal.get("proposed_action", "Adjust strategy based on detected pattern")
    changes_made: list[str] = []

    # 1) Seniority adjustment when explicitly suggested by proposal action.
    action_lower = action.lower()
    if any(token in action_lower for token in ["down-level", "downlevel", "seniority", "adjust target seniority"]):
        old_seniority = strategy.get("target_seniority", "Senior")
        new_seniority = _downlevel_seniority(old_seniority)
        if new_seniority != old_seniority:
            strategy["target_seniority"] = new_seniority
            changes_made.append(f"Target seniority changed from {old_seniority} to {new_seniority}")

    # 2) Prep recommendations are always updated with theme-relevant resources.
    prep = PREP_RESOURCES.get(theme, [f"Study targeted resources for {theme}"])
    existing_prep = strategy.get("prep_recommendations", [])
    new_prep = [p for p in prep if p not in existing_prep]
    strategy["prep_recommendations"] = existing_prep + new_prep
    if new_prep:
        changes_made.append(f"Added {len(new_prep)} preparation recommendation(s) for {theme}")

    # 3) Focus area update for skill patterns.
    focus = list(strategy.get("focus_areas", []))
    derived_focus = theme.replace(" Skills", "")
    if derived_focus not in focus and theme in ("System Design", "Frontend Skills", "Backend Skills"):
        focus.append(derived_focus)
        strategy["focus_areas"] = focus
        changes_made.append(f"Added '{derived_focus}' to focus areas")

    # 4) Guarantee at least one visible strategy parameter change.
    if not changes_made:
        fallback_note = f"Targeted follow-up for pattern: {theme}"
        strategy["prep_recommendations"] = strategy.get("prep_recommendations", []) + [fallback_note]
        changes_made.append("Added explicit preparation follow-up note")

    # Lock this theme until cooldown is satisfied.
    locked = list(strategy.get("locked_changes", []))
    if theme not in locked:
        locked.append(theme)
    strategy["locked_changes"] = locked
    strategy["positive_events_since_last_change"] = 0

    strategy_change_log = list(state.get("strategy_change_log", []))
    strategy_change_log.append(
        {
            "date": datetime.now().strftime("%Y-%m-%d"),
            "theme": theme,
            "proposed_action": action,
            "changes": changes_made,
        }
    )

    return {
        "current_strategy": strategy,
        "strategy_change_log": strategy_change_log,
        "strategy_approved": False,
        "strategy_proposal": {},
    }


## `src/profile_builder.py`


In [ ]:
"""
Profile creation and document-ingestion utilities.
Supports creating user profiles from uploaded CV/job documents with optional Gemini parsing.
"""

from __future__ import annotations

import json
import os
import re
from io import BytesIO
from typing import Any


DATA_DIR = os.path.join(os.path.dirname(__file__), "..", "data")
DEMO_PROFILES_PATH = os.path.join(DATA_DIR, "demo_profiles.json")
USER_PROFILES_PATH = os.path.join(DATA_DIR, "user_profiles.json")

TEXT_EXTENSIONS = {
    ".txt",
    ".md",
    ".csv",
    ".json",
    ".yaml",
    ".yml",
    ".log",
    ".rst",
}

SKILL_KEYWORDS = [
    "python",
    "java",
    "javascript",
    "typescript",
    "react",
    "node.js",
    "node",
    "aws",
    "docker",
    "kubernetes",
    "postgresql",
    "mysql",
    "mongodb",
    "rest api",
    "graphql",
    "system design",
    "microservices",
    "django",
    "flask",
    "spring",
    "terraform",
    "ci/cd",
    "git",
]


def _empty_store() -> dict[str, dict[str, Any]]:
    return {"profiles": {}, "default_strategies": {}}


def _ensure_store_shape(data: dict[str, Any]) -> dict[str, dict[str, Any]]:
    if not isinstance(data, dict):
        return _empty_store()
    data.setdefault("profiles", {})
    data.setdefault("default_strategies", {})
    return data


def _slugify(name: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    return slug or "candidate"


def _seniority_from_years(years: int) -> str:
    if years <= 2:
        return "Junior"
    if years <= 5:
        return "Mid-Level"
    return "Senior"


def _extract_skills_keyword(text: str) -> list[str]:
    text_lower = text.lower()
    hits = []
    for skill in SKILL_KEYWORDS:
        if skill in text_lower:
            hits.append(skill.title() if skill != "ci/cd" else "CI/CD")
    # Preserve order while deduping.
    unique = []
    seen = set()
    for skill in hits:
        if skill not in seen:
            seen.add(skill)
            unique.append(skill)
    return unique


def _extract_salary_keyword(text: str) -> str:
    """
    Best-effort salary extraction from free text.
    Examples: "$180k", "180000", "120k-150k", "INR 30 LPA".
    """
    if not text:
        return ""
    patterns = [
        r"(\$\s?\d{2,3}(?:,\d{3})?(?:\s?[kK])?(?:\s?-\s?\$?\s?\d{2,3}(?:,\d{3})?(?:\s?[kK])?)?)",
        r"((?:usd|inr|eur)\s?\d[\d,]*(?:\s?-\s?\d[\d,]*)?)",
        r"(\d{2,3}\s?[kK](?:\s?-\s?\d{2,3}\s?[kK])?)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return ""


def _default_strategy(seniority: str, base_skills: list[str]) -> dict[str, Any]:
    return {
        "target_seniority": seniority,
        "focus_areas": base_skills[:3],
        "prep_recommendations": [],
        "locked_changes": [],
        "positive_events_since_last_change": 0,
    }


def _strip_json_fence(content: str) -> str:
    cleaned = content.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?", "", cleaned).strip()
        cleaned = re.sub(r"```$", "", cleaned).strip()
    return cleaned


def _extract_text_from_pdf(file_bytes: bytes) -> str:
    try:
        from PyPDF2 import PdfReader
    except Exception:
        return ""

    try:
        reader = PdfReader(BytesIO(file_bytes))
        pages = [page.extract_text() or "" for page in reader.pages]
        return "\n".join(pages).strip()
    except Exception:
        return ""


def _extract_upload_text(uploaded_file: Any) -> tuple[str, str]:
    """
    Returns:
      (text, warning_message)
    """
    name = getattr(uploaded_file, "name", "uploaded_file")
    ext = os.path.splitext(name)[1].lower()
    raw = uploaded_file.getvalue()

    if ext in TEXT_EXTENSIONS:
        return raw.decode("utf-8", errors="ignore"), ""

    if ext == ".pdf":
        text = _extract_text_from_pdf(raw)
        if text:
            return text, ""
        return "", f"Could not parse PDF text from {name} (install PyPDF2 or upload as .txt)."

    return "", f"Unsupported file type for text extraction: {name}"


def _llm_extract_profile(
    *,
    raw_context: str,
    fallback_name: str,
    fallback_role: str,
    fallback_years: int,
    fallback_salary: str,
    api_key: str,
) -> dict[str, Any] | None:
    if not api_key:
        return None

    try:
        from langchain_google_genai import ChatGoogleGenerativeAI

        llm = ChatGoogleGenerativeAI(
            model="gemini-2.0-flash",
            google_api_key=api_key,
            temperature=0,
        )

        prompt = (
            "Extract a structured candidate profile from the context below.\n"
            "Return JSON only (no prose) with keys:\n"
            "name (string), target_role (string), years_experience (int),\n"
            "base_skills (array of strings), preferred_job_types (array of strings),\n"
            "target_salary (string).\n"
            "Limit base_skills to 20 concrete technical/professional skills.\n\n"
            f"Fallback name: {fallback_name}\n"
            f"Fallback role: {fallback_role}\n"
            f"Fallback years: {fallback_years}\n\n"
            f"Fallback salary: {fallback_salary}\n\n"
            f"Context:\n{raw_context[:14000]}"
        )

        response = llm.invoke(prompt)
        content = _strip_json_fence(response.content if hasattr(response, "content") else str(response))
        parsed = json.loads(content)
        if isinstance(parsed, dict):
            return parsed
        return None
    except Exception:
        return None


def load_all_profiles() -> dict[str, dict[str, Any]]:
    with open(DEMO_PROFILES_PATH, "r", encoding="utf-8") as handle:
        demo = _ensure_store_shape(json.load(handle))

    if os.path.exists(USER_PROFILES_PATH):
        with open(USER_PROFILES_PATH, "r", encoding="utf-8") as handle:
            user = _ensure_store_shape(json.load(handle))
    else:
        user = _empty_store()

    merged_profiles = dict(demo["profiles"])
    merged_profiles.update(user["profiles"])
    merged_strategies = dict(demo["default_strategies"])
    merged_strategies.update(user["default_strategies"])

    return {"profiles": merged_profiles, "default_strategies": merged_strategies}


def create_or_update_user_profile(
    *,
    name: str,
    target_role: str,
    years_experience: int,
    target_salary: str,
    additional_notes: str,
    uploaded_files: list[Any],
    gemini_api_key: str = "",
) -> tuple[str, dict[str, Any], dict[str, Any], list[str]]:
    warnings: list[str] = []
    effective_api_key = (gemini_api_key or os.environ.get("GOOGLE_API_KEY", "")).strip()

    extracted_docs: list[str] = []
    for uploaded in uploaded_files:
        text, warning = _extract_upload_text(uploaded)
        if text.strip():
            extracted_docs.append(f"FILE: {uploaded.name}\n{text.strip()[:8000]}")
        elif warning:
            warnings.append(warning)

    combined_context = "\n\n".join(extracted_docs + [additional_notes.strip()])
    llm_profile = _llm_extract_profile(
        raw_context=combined_context,
        fallback_name=name,
        fallback_role=target_role,
        fallback_years=years_experience,
        fallback_salary=target_salary,
        api_key=effective_api_key,
    )

    keyword_skills = _extract_skills_keyword(combined_context)
    llm_skills = llm_profile.get("base_skills", []) if llm_profile else []

    merged_skills = []
    for skill in llm_skills + keyword_skills:
        skill_str = str(skill).strip()
        if skill_str and skill_str not in merged_skills:
            merged_skills.append(skill_str)

    final_name = str(llm_profile.get("name", "")).strip() if llm_profile else ""
    final_role = str(llm_profile.get("target_role", "")).strip() if llm_profile else ""
    final_years = llm_profile.get("years_experience", years_experience) if llm_profile else years_experience
    final_salary = str(llm_profile.get("target_salary", "")).strip() if llm_profile else ""
    try:
        final_years = int(final_years)
    except Exception:
        final_years = years_experience

    salary_fallback = target_salary.strip() or _extract_salary_keyword(combined_context)

    profile = {
        "name": final_name or name.strip(),
        "base_skills": merged_skills[:20],
        "target_role": final_role or target_role.strip(),
        "years_experience": max(0, final_years),
        "target_salary": final_salary or salary_fallback,
        "notes": additional_notes.strip(),
    }

    seniority = _seniority_from_years(profile["years_experience"])
    strategy = _default_strategy(seniority, profile["base_skills"])

    profile_key = f"{_slugify(profile['name'])}_user"

    if os.path.exists(USER_PROFILES_PATH):
        with open(USER_PROFILES_PATH, "r", encoding="utf-8") as handle:
            user_store = _ensure_store_shape(json.load(handle))
    else:
        user_store = _empty_store()

    user_store["profiles"][profile_key] = profile
    user_store["default_strategies"][profile_key] = strategy

    with open(USER_PROFILES_PATH, "w", encoding="utf-8") as handle:
        json.dump(user_store, handle, indent=2)

    if not llm_profile:
        warnings.append("Gemini extraction unavailable; used keyword and manual inputs.")
    if not profile["base_skills"]:
        warnings.append("No skills extracted. Add more detail in notes or upload a text/PDF resume.")

    return profile_key, profile, strategy, warnings


## `src/utils.py`


In [ ]:
﻿"""
Utility helpers for JobHunter - scoring, keyword overlap, KPI computation.
"""

from __future__ import annotations

from datetime import datetime
from typing import List


DATE_FMT = "%Y-%m-%d"


def compute_skill_overlap(job_skills: List[str], candidate_skills: List[str]) -> float:
    """Calculate percentage overlap between job requirements and candidate skills."""
    if not job_skills:
        return 50.0

    job_set = {s.strip().lower() for s in job_skills}
    cand_set = {s.strip().lower() for s in candidate_skills}

    if not job_set:
        return 50.0

    overlap = job_set & cand_set
    return round((len(overlap) / len(job_set)) * 100, 1)


def apply_seniority_penalty(score: float, job_seniority: str, target_seniority: str) -> float:
    """Drop score when job seniority does not match strategy target."""
    if job_seniority.strip().lower() != target_seniority.strip().lower():
        return round(score * 0.1, 1)
    return score


def _safe_days_between(start_date: str, end_date: str) -> int | None:
    """Return day delta for YYYY-MM-DD dates, otherwise None."""
    try:
        start = datetime.strptime(start_date, DATE_FMT)
        end = datetime.strptime(end_date, DATE_FMT)
        return max((end - start).days, 0)
    except Exception:
        return None


def compute_kpi_metrics(applications: list, usefulness_ratings: list | None = None) -> dict:
    """Compute business KPIs from applications and candidate usefulness ratings."""
    total = len(applications)

    counts = {
        "applied": 0,
        "screening": 0,
        "interviewing": 0,
        "rejected": 0,
        "offer": 0,
    }

    outcome_days: list[int] = []

    for app in applications:
        status = app.get("status", "applied") if isinstance(app, dict) else app.status
        if status in counts:
            counts[status] += 1

        if isinstance(app, dict) and status in ("rejected", "offer"):
            applied_date = app.get("date_applied", "")
            outcome_date = app.get("date_outcome", "") or app.get("date_last_updated", "")
            days = _safe_days_between(applied_date, outcome_date)
            if days is not None:
                outcome_days.append(days)

    interview_eligible = counts["interviewing"] + counts["offer"]
    interview_rate = (interview_eligible / total * 100) if total > 0 else 0.0
    offer_rate = (counts["offer"] / max(interview_eligible, 1) * 100) if interview_eligible > 0 else 0.0

    avg_outcome_days = round(sum(outcome_days) / len(outcome_days), 1) if outcome_days else None

    ratings = usefulness_ratings or []
    usefulness_avg = round(sum(ratings) / len(ratings), 2) if ratings else None

    return {
        "total_applications": total,
        **counts,
        "interview_conversion_rate": round(interview_rate, 1),
        "offer_rate": round(offer_rate, 1),
        "avg_days_to_outcome": avg_outcome_days,
        "pattern_usefulness_avg": usefulness_avg,
        "pattern_usefulness_count": len(ratings),
    }


def normalize_skills(skills_str: str) -> List[str]:
    """Split a comma-separated skills string into a cleaned list."""
    if not skills_str:
        return []
    return [s.strip() for s in skills_str.split(",") if s.strip()]


## `ui/__init__.py`


In [ ]:
# JobHunter UI package


## `ui/applications_panel.py`


In [ ]:
"""
Applications panel for the dashboard.
Tracks application progress and captures rejection feedback.
"""

from __future__ import annotations

from collections import Counter
from datetime import datetime

import streamlit as st

from ui.components import render_metric_card, render_status_badge


STATUSES = ["applied", "screening", "interviewing", "rejected", "offer"]


def _compute_kpi(applications: list[dict]) -> dict:
    counts = Counter(app.get("status", "applied") for app in applications)
    total = len(applications)
    interview_count = counts.get("interviewing", 0) + counts.get("offer", 0)
    conversion = round((interview_count / total) * 100, 1) if total else 0
    return {
        "total_applications": total,
        "interviewing": counts.get("interviewing", 0),
        "interview_conversion_rate": conversion,
        "offer": counts.get("offer", 0),
    }


def render_applications_panel(state: dict):
    """Render the applications tracker panel."""
    st.markdown('<div class="section-header">Application Tracker</div>', unsafe_allow_html=True)
    st.markdown(
        '<div class="sub-header">Update statuses quickly and log rejection context for the next strategy run.</div>',
        unsafe_allow_html=True,
    )

    applications = list(state.get("applications", []))

    pending_apps = st.session_state.get("new_applications", [])
    if pending_apps:
        st.info(f"{len(pending_apps)} new application(s) waiting to be processed by the next agent run.")

    if not applications:
        st.info("No applications tracked yet. Add one from Recommended Jobs.")
        return

    kpi = state.get("kpi_metrics") or _compute_kpi(applications)

    m1, m2, m3, m4 = st.columns(4)
    with m1:
        st.markdown(render_metric_card(str(kpi.get("total_applications", 0)), "Total", "#1D4ED8"), unsafe_allow_html=True)
    with m2:
        st.markdown(render_metric_card(str(kpi.get("interviewing", 0)), "Interviewing", "#0F766E"), unsafe_allow_html=True)
    with m3:
        st.markdown(
            render_metric_card(f"{kpi.get('interview_conversion_rate', 0)}%", "Interview Rate", "#B45309"),
            unsafe_allow_html=True,
        )
    with m4:
        st.markdown(render_metric_card(str(kpi.get("offer", 0)), "Offers", "#15803D"), unsafe_allow_html=True)

    counts = Counter(app.get("status", "applied") for app in applications)
    status_options = ["all"] + STATUSES
    selected_status = st.radio(
        "Filter by status",
        options=status_options,
        horizontal=True,
        format_func=lambda x: f"{x.title()} ({len(applications) if x == 'all' else counts.get(x, 0)})",
    )

    sort_choice = st.selectbox("Sort", options=["Newest first", "Oldest first", "Company A-Z"])

    filtered = applications
    if selected_status != "all":
        filtered = [a for a in filtered if a.get("status") == selected_status]

    if sort_choice == "Newest first":
        filtered = sorted(filtered, key=lambda x: x.get("date_applied", ""), reverse=True)
    elif sort_choice == "Oldest first":
        filtered = sorted(filtered, key=lambda x: x.get("date_applied", ""))
    else:
        filtered = sorted(filtered, key=lambda x: x.get("company_name", "").lower())

    st.caption(f"Showing {len(filtered)} of {len(applications)} applications")

    for i, app in enumerate(filtered):
        status = app.get("status", "applied")

        st.markdown(
            f"""
            <div class="job-card">
                <div style="display:flex; justify-content:space-between; align-items:flex-start; gap:10px;">
                    <div>
                        <div class="job-title">{app.get('role_title', 'Unknown Role')}</div>
                        <div class="job-company">{app.get('company_name', 'Unknown Company')}</div>
                        <div style="font-size:12px; color:#64748B; margin-top:4px;">
                            Applied: {app.get('date_applied', 'N/A')} | Job ID: {app.get('job_id', 'N/A')[:8]}
                        </div>
                    </div>
                    <div>{render_status_badge(status)}</div>
                </div>
            </div>
            """,
            unsafe_allow_html=True,
        )

        c1, c2 = st.columns([1, 1.2])
        with c1:
            new_status = st.selectbox(
                "Status",
                STATUSES,
                index=STATUSES.index(status) if status in STATUSES else 0,
                key=f"status_update_{app.get('job_id', i)}_{i}",
            )
            if st.button("Save status", key=f"save_status_{app.get('job_id', i)}_{i}", use_container_width=True):
                app["status"] = new_status
                app["date_last_updated"] = datetime.now().strftime("%Y-%m-%d")
                if new_status in ("rejected", "offer"):
                    app["date_outcome"] = app["date_last_updated"]
                st.session_state.persistent_state["applications"] = applications
                st.success("Status updated.")
                st.rerun()

        with c2:
            if status == "rejected":
                feedback = st.text_area(
                    "Rejection note",
                    placeholder="Paste rejection reason or summary...",
                    key=f"feedback_{app.get('job_id', i)}_{i}",
                    height=80,
                )
                if st.button("Log feedback", key=f"log_fb_{app.get('job_id', i)}_{i}", use_container_width=True):
                    if feedback.strip():
                        st.session_state.manual_feedback.append(
                            {
                                "app_id": app.get("job_id", ""),
                                "text": feedback.strip(),
                            }
                        )
                        st.success("Feedback added. Run agent to parse patterns.")
                        st.rerun()
                    else:
                        st.warning("Add feedback text before logging.")

    feedback_log = state.get("feedback_log", [])
    if feedback_log:
        st.markdown("---")
        st.markdown("**Recent Parsed Feedback**")
        for fb in feedback_log[-5:]:
            st.markdown(
                f"""
                <div class="strategy-item">
                    <strong>{fb.get('extracted_reason', 'Unknown reason')}</strong><br>
                    <span style="font-size:11px; color:#64748B;">
                        App: {fb.get('application_id', 'N/A')[:8]} | {fb.get('date_logged', '')}
                    </span>
                </div>
                """,
                unsafe_allow_html=True,
            )



## `ui/components.py`


In [ ]:
"""
Shared UI components for the JobHunter Streamlit dashboard.
Status badges, metric cards, and styling utilities.
"""

from __future__ import annotations

import streamlit as st


STATUS_COLORS = {
    "applied": "#2563EB",
    "screening": "#D97706",
    "interviewing": "#0F766E",
    "rejected": "#B91C1C",
    "offer": "#15803D",
}

STATUS_ICONS = {
    "applied": "Submitted",
    "screening": "Screening",
    "interviewing": "Interview",
    "rejected": "Rejected",
    "offer": "Offer",
}

SENIORITY_COLORS = {
    "Junior": "#15803D",
    "Mid-Level": "#B45309",
    "Senior": "#1D4ED8",
}


def inject_custom_css():
    """Inject custom CSS for dashboard styling."""
    st.markdown(
        """
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;700&family=IBM+Plex+Sans:wght@400;500;600&display=swap');

    .stApp {
        font-family: 'IBM Plex Sans', sans-serif;
        background:
            radial-gradient(circle at 8% 10%, rgba(14, 165, 233, 0.10) 0, transparent 38%),
            radial-gradient(circle at 88% 12%, rgba(249, 115, 22, 0.10) 0, transparent 32%),
            linear-gradient(180deg, #F8FAFC 0%, #F1F5F9 100%);
        color: #0F172A;
    }

    h1, h2, h3, .section-header, .brand-title {
        font-family: 'Space Grotesk', sans-serif;
        letter-spacing: -0.01em;
    }

    .panel-card {
        border: 1px solid #D7DEE8;
        background: #FFFFFF;
        border-radius: 16px;
        padding: 16px;
        box-shadow: 0 10px 24px rgba(15, 23, 42, 0.05);
    }

    .job-card {
        background: #FFFFFF;
        border: 1px solid #D7DEE8;
        border-radius: 14px;
        padding: 14px;
        margin-bottom: 10px;
        transition: border-color 0.2s ease, box-shadow 0.2s ease;
    }

    .job-card:hover {
        border-color: #0EA5E9;
        box-shadow: 0 8px 18px rgba(14, 165, 233, 0.12);
    }

    .job-title {
        font-size: 16px;
        font-weight: 600;
        color: #0F172A;
        margin-bottom: 4px;
    }

    .job-company {
        font-size: 13px;
        color: #475569;
        margin-bottom: 6px;
    }

    .fit-score-bar {
        height: 8px;
        border-radius: 999px;
        background: #E2E8F0;
        overflow: hidden;
        margin-top: 8px;
    }

    .fit-score-fill {
        height: 100%;
        border-radius: 999px;
        transition: width 0.35s ease;
    }

    .status-badge {
        display: inline-block;
        padding: 4px 10px;
        border-radius: 999px;
        font-size: 11px;
        font-weight: 600;
        color: #FFFFFF;
    }

    .seniority-badge {
        display: inline-block;
        padding: 2px 8px;
        border-radius: 8px;
        font-size: 11px;
        font-weight: 600;
        color: #FFFFFF;
    }

    .skill-chip {
        display: inline-block;
        background: #F1F5F9;
        border: 1px solid #D1D9E6;
        padding: 2px 8px;
        border-radius: 999px;
        font-size: 11px;
        color: #334155;
        margin: 2px 4px 2px 0;
    }

    .metric-card {
        background: #FFFFFF;
        border: 1px solid #D7DEE8;
        border-radius: 12px;
        padding: 12px;
        text-align: center;
    }

    .metric-value {
        font-size: 24px;
        font-weight: 700;
        color: #0F172A;
    }

    .metric-label {
        font-size: 12px;
        color: #64748B;
        margin-top: 2px;
    }

    .alert-box {
        background: #FEF2F2;
        border: 1px solid #FECACA;
        border-radius: 12px;
        padding: 14px;
        margin: 10px 0;
    }

    .strategy-item {
        background: #F8FAFC;
        border: 1px solid #E2E8F0;
        border-radius: 10px;
        padding: 10px 12px;
        margin-bottom: 8px;
    }

    .section-header {
        font-size: 20px;
        font-weight: 700;
        color: #0F172A;
        margin-bottom: 10px;
    }

    .sub-header {
        font-size: 12px;
        color: #475569;
        margin-bottom: 14px;
    }

    .brand-wrap {
        text-align: center;
        padding: 8px 0 14px 0;
    }

    .brand-mark {
        width: 38px;
        height: 38px;
        border-radius: 10px;
        margin: 0 auto;
        background: linear-gradient(145deg, #0369A1 0%, #EA580C 100%);
    }

    .brand-title {
        font-size: 22px;
        font-weight: 700;
        color: #0F172A;
        margin-top: 8px;
    }

    .brand-sub {
        font-size: 12px;
        color: #64748B;
    }

    .next-step {
        border-left: 3px solid #0EA5E9;
        background: #F0F9FF;
        border-radius: 8px;
        padding: 8px 10px;
        margin-bottom: 8px;
        font-size: 13px;
        color: #0F172A;
    }
    </style>
    """,
        unsafe_allow_html=True,
    )


def render_status_badge(status: str) -> str:
    """Return HTML for a colored status badge."""
    color = STATUS_COLORS.get(status, "#64748B")
    label = STATUS_ICONS.get(status, status.title())
    return f'<span class="status-badge" style="background: {color};">{label}</span>'


def render_seniority_badge(seniority: str) -> str:
    """Return HTML for a seniority level badge."""
    color = SENIORITY_COLORS.get(seniority, "#64748B")
    return f'<span class="seniority-badge" style="background: {color};">{seniority}</span>'


def render_fit_score_bar(score: float) -> str:
    """Return HTML for a fit score progress bar."""
    if score >= 70:
        color = "#15803D"
    elif score >= 40:
        color = "#B45309"
    else:
        color = "#B91C1C"

    return f"""
    <div class="fit-score-bar">
        <div class="fit-score-fill" style="width: {min(score, 100)}%; background: {color};"></div>
    </div>
    """


def render_skill_chips(skills: list) -> str:
    """Return HTML for skill chips."""
    chips = "".join(f'<span class="skill-chip">{s}</span>' for s in skills[:6])
    extra = len(skills) - 6
    if extra > 0:
        chips += f'<span class="skill-chip">+{extra} more</span>'
    return chips


def render_metric_card(value: str, label: str, color: str = "#0369A1") -> str:
    """Return HTML for a KPI metric card."""
    return f"""
    <div class="metric-card">
        <div class="metric-value" style="color: {color};">{value}</div>
        <div class="metric-label">{label}</div>
    </div>
    """


## `ui/jobs_panel.py`


In [ ]:
"""
Jobs panel for the dashboard.
Shows ranked jobs with filters and one-click apply actions.
"""

from __future__ import annotations

from datetime import datetime

import streamlit as st

from ui.components import render_fit_score_bar, render_seniority_badge, render_skill_chips


def render_jobs_panel(state: dict):
    """Render the job feed panel."""
    st.markdown('<div class="section-header">Recommended Jobs</div>', unsafe_allow_html=True)
    st.markdown(
        '<div class="sub-header">Filter matches, inspect fit, then add applications in one click.</div>',
        unsafe_allow_html=True,
    )

    jobs = state.get("matched_jobs", [])
    strategy = state.get("current_strategy", {})

    if not jobs:
        st.info("Run the agent to fetch personalized job recommendations.")
        return

    target = strategy.get("target_seniority", "Senior")
    st.caption(f"Current target: {target}")

    all_seniorities = sorted({j.get("seniority", "Unknown") for j in jobs})

    c1, c2, c3 = st.columns([1, 1, 1.2])
    with c1:
        min_score = st.slider("Minimum fit", min_value=0, max_value=100, value=45, step=5)
    with c2:
        seniority_filter = st.multiselect(
            "Seniority",
            options=all_seniorities,
            default=all_seniorities,
        )
    with c3:
        search_term = st.text_input("Keyword", placeholder="backend, python, fintech...").strip().lower()

    existing_apps = state.get("applications", [])
    existing_ids = {app.get("job_id") for app in existing_apps}
    pending_ids = {app.get("job_id") for app in st.session_state.get("new_applications", [])}

    filtered_jobs = []
    for job in jobs:
        score = float(job.get("fit_score", 0))
        seniority = job.get("seniority", "Unknown")
        haystack = " ".join(
            [
                str(job.get("title", "")),
                str(job.get("company", "")),
                str(job.get("description", "")),
                " ".join(job.get("required_skills", [])[:10]),
            ]
        ).lower()

        if score < min_score:
            continue
        if seniority_filter and seniority not in seniority_filter:
            continue
        if search_term and search_term not in haystack:
            continue
        filtered_jobs.append(job)

    st.caption(f"Showing {len(filtered_jobs)} of {len(jobs)} jobs")

    if not filtered_jobs:
        st.warning("No jobs match these filters. Lower the minimum fit or clear the keyword.")
        return

    for i, job in enumerate(filtered_jobs):
        score = float(job.get("fit_score", 0))
        skills = job.get("required_skills", [])
        score_color = "#15803D" if score >= 70 else "#B45309" if score >= 40 else "#B91C1C"
        seniority_html = render_seniority_badge(job.get("seniority", "Unknown"))
        skills_html = render_skill_chips(skills)
        score_bar_html = render_fit_score_bar(score)

        st.markdown(
            f"""
            <div class="job-card">
                <div style="display:flex; justify-content:space-between; gap:10px; align-items:flex-start;">
                    <div>
                        <div class="job-title">{job.get('title', 'Untitled Role')}</div>
                        <div class="job-company">{job.get('company', 'Unknown Company')} | {seniority_html}</div>
                    </div>
                    <div style="font-size:13px; font-weight:700; color:{score_color};">{int(score)}% fit</div>
                </div>
                <div style="margin-top:6px;">{skills_html}</div>
                <div style="margin-top:8px;">{score_bar_html}</div>
            </div>
            """,
            unsafe_allow_html=True,
        )

        job_id = job.get("job_id", str(i))
        already_added = job_id in existing_ids or job_id in pending_ids

        b1, b2 = st.columns([1, 1.2])
        with b1:
            if st.button(
                "Applied" if already_added else "Add Application",
                key=f"apply_{job_id}_{i}",
                use_container_width=True,
                disabled=already_added,
            ):
                if "new_applications" not in st.session_state:
                    st.session_state.new_applications = []

                st.session_state.new_applications.append(
                    {
                        "job_id": job_id,
                        "company_name": job.get("company", ""),
                        "role_title": job.get("title", ""),
                        "status": "applied",
                        "date_applied": datetime.now().strftime("%Y-%m-%d"),
                        "date_last_updated": datetime.now().strftime("%Y-%m-%d"),
                        "date_outcome": "",
                    }
                )
                st.success(f"Added {job.get('title', '')} at {job.get('company', '')} to applications queue.")
                st.rerun()

        with b2:
            description = (job.get("description") or "").strip()
            with st.expander("Show details"):
                st.write(description if description else "No job description available.")



## `ui/strategy_panel.py`


In [ ]:
"""
Strategy panel for the dashboard.
Explains the active strategy, detected patterns, and pending strategy decisions.
"""

from __future__ import annotations

import streamlit as st

from ui.components import render_seniority_badge


def render_strategy_panel(state: dict):
    """Render the strategy and pattern insights panel."""
    st.markdown('<div class="section-header">Agent Strategy Log</div>', unsafe_allow_html=True)
    st.markdown(
        '<div class="sub-header">Review what the agent learned and approve or reject strategy pivots.</div>',
        unsafe_allow_html=True,
    )

    strategy = state.get("current_strategy", {})
    patterns = state.get("rejection_patterns", [])
    proposal = state.get("strategy_proposal", {})
    session_count = state.get("session_count", state.get("sessions_count", 0))
    strategy_change_log = state.get("strategy_change_log", [])
    usefulness_ratings = state.get("usefulness_ratings", [])

    target_seniority = strategy.get("target_seniority", "Senior")
    st.markdown(
        f"""
        <div class="job-card">
            <div style="font-size:12px; color:#64748B;">Current direction</div>
            <div style="font-size:22px; font-weight:700; color:#0F172A; margin-top:4px;">
                {render_seniority_badge(target_seniority)} {target_seniority} roles
            </div>
            <div style="font-size:12px; color:#64748B; margin-top:8px;">Completed sessions: {session_count}</div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    focus = strategy.get("focus_areas", [])
    if focus:
        st.markdown("**Focus Areas**")
        for area in focus:
            st.markdown(f'<div class="strategy-item">{area}</div>', unsafe_allow_html=True)

    prep = strategy.get("prep_recommendations", [])
    if prep:
        st.markdown("**Preparation Plan**")
        for rec in prep:
            st.markdown(f'<div class="strategy-item">{rec}</div>', unsafe_allow_html=True)

    st.markdown("---")
    st.markdown("**Detected Patterns**")

    if not patterns:
        feedback_count = len(state.get("feedback_log", []))
        if feedback_count < 3:
            st.info(
                f"Pattern detection activates after 3 feedback entries. "
                f"Current progress: {feedback_count}/3."
            )
        else:
            st.success("No repeated negative pattern detected yet.")
    else:
        for pattern in patterns:
            occurrences = pattern.get("occurrences", 0)
            severity = "High" if occurrences >= 5 else "Medium"
            color = "#B91C1C" if occurrences >= 5 else "#B45309"
            st.markdown(
                f"""
                <div class="alert-box">
                    <div style="font-size:15px; font-weight:700; color:{color};">
                        {pattern.get('theme', 'Unknown Pattern')} ({severity})
                    </div>
                    <div style="font-size:13px; color:#334155; margin-top:6px;">
                        Seen {occurrences} time(s) in rejection feedback.
                    </div>
                    <div style="font-size:12px; color:#475569; margin-top:8px;">
                        Suggested action: {pattern.get('proposed_action', 'Review and adjust strategy')}
                    </div>
                </div>
                """,
                unsafe_allow_html=True,
            )

    if proposal and proposal.get("theme"):
        st.markdown("---")
        st.markdown("### Pending Agent Recommendation")
        st.markdown(
            f"""
            <div class="job-card" style="border-color:#7DD3FC; background:#F0F9FF;">
                <div style="font-size:15px; font-weight:700; color:#0C4A6E;">Strategy Pivot Proposed</div>
                <div style="font-size:13px; color:#334155; margin-top:8px;">
                    Theme: <strong>{proposal.get('theme', '')}</strong> | Occurrences: {proposal.get('occurrences', 0)}
                </div>
                <div style="font-size:13px; color:#334155; margin-top:6px;">
                    Proposed change: {proposal.get('proposed_action', '')}
                </div>
            </div>
            """,
            unsafe_allow_html=True,
        )

        a1, a2 = st.columns(2)
        with a1:
            if st.button("Approve Pivot", key="approve_strategy", type="primary", use_container_width=True):
                st.session_state.strategy_approved = True
                st.success("Pivot approved. Run the agent to apply this strategy change.")
        with a2:
            if st.button("Reject Pivot", key="reject_strategy", use_container_width=True):
                st.session_state.strategy_approved = False
                st.session_state.strategy_rejected = True
                st.info("Pivot rejected. Current strategy remains active.")

    locked = strategy.get("locked_changes", [])
    if locked:
        st.markdown("---")
        st.markdown("**Cooldown Changes**")
        positive_count = strategy.get("positive_events_since_last_change", 0)
        for change in locked:
            st.markdown(
                f'<div class="strategy-item">{change} (unlock progress: {positive_count}/3 positive events)</div>',
                unsafe_allow_html=True,
            )

    if strategy_change_log:
        st.markdown("---")
        st.markdown("**Strategy Change History**")
        for entry in reversed(strategy_change_log[-3:]):
            changes = "; ".join(entry.get("changes", []))
            st.markdown(
                f"""
                <div class="strategy-item">
                    <strong>{entry.get('date', '')} - {entry.get('theme', 'Pattern')}</strong><br>
                    <span style="font-size:12px; color:#334155;">{entry.get('proposed_action', '')}</span><br>
                    <span style="font-size:11px; color:#64748B;">Applied changes: {changes}</span>
                </div>
                """,
                unsafe_allow_html=True,
            )

    st.markdown("---")
    st.markdown("**Pattern Detection Usefulness**")
    avg_usefulness = state.get("kpi_metrics", {}).get("pattern_usefulness_avg")
    if avg_usefulness is None and usefulness_ratings:
        avg_usefulness = round(sum(usefulness_ratings) / len(usefulness_ratings), 2)
    if avg_usefulness is not None:
        st.caption(f"Average usefulness rating: {avg_usefulness}/5 from {len(usefulness_ratings)} response(s)")
    else:
        st.caption("No usefulness ratings yet.")

    rating = st.slider(
        "How useful were these pattern insights this session?",
        min_value=1,
        max_value=5,
        value=3,
        key="pattern_usefulness_slider",
    )
    if st.button("Save usefulness rating", use_container_width=True):
        ratings = list(st.session_state.persistent_state.get("usefulness_ratings", []))
        ratings.append(rating)
        st.session_state.persistent_state["usefulness_ratings"] = ratings
        st.success("Usefulness rating saved.")
        st.rerun()

